# AI Video Assistant - Qwen3 LLM Analysis (Kaggle)

Notebook này được thiết kế để phân tích kịch bản thành thẻ meta-data bằng cấu trúc LLM `Qwen/Qwen2.5-7B-Instruct` mạnh mẽ.

## 1. Yêu cầu phần cứng
- Bật Accelerator T4 x2 hoặc P100.
- Upload file `analysis_requests.json` (tạo ra bởi code Local) vào cùng thư mục với notebook này (`/kaggle/working/`).

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json
import os
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Đọc dữ liệu mồi từ dưới máy local gửi lên
requests_path = "analysis_requests.json"
if not os.path.exists(requests_path):
    raise FileNotFoundError(f"Vui lòng upload {requests_path} trước khi chạy!")

with open(requests_path, "r", encoding="utf-8") as f:
    requests = json.load(f)

print(f"Đã đọc {len(requests)} bản ghi kịch bản cần phân tích.")

In [ ]:
# 2. Khởi tạo Mô hình ngôn ngữ Qwen (Phiên bản Instruct)
model_id = "Qwen/Qwen2.5-7B-Instruct"

print("Đang tải mô hình (có thể mất vài phút)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
)
print("Tải mô hình thành công!")

In [ ]:
# 3. Hàm truy vấn AI
def analyze_segment_with_llm(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.3, # Để nhiệt độ thấp giúp câu trả lời chuẩn JSON và logic hơn
        do_sample=True
    )
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# 4. Hàm trích xuất JSON
def extract_json(response):
    # Tìm kiếm khối mã json trong văn bản
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    # Hoặc cố gắng parse toàn bộ nếu nó chỉ return json
    try:
        return json.loads(response)
    except:
        return None

In [ ]:
# 5. Xử lý toàn bộ dữ liệu
results = []

for req in requests:
    print(f"\nĐang phân tích Segment ID: {req['segment_id']}...")
    response_text = analyze_segment_with_llm(req['messages'])
    
    parsed_data = extract_json(response_text)
    if parsed_data is None:
        print(f"⚠️ Cảnh báo: AI không trả về chuẩn JSON cho segment {req['segment_id']}. Dùng dữ liệu rỗng.")
        parsed_data = {
            "keywords": ["error"],
            "scene": "general",
            "tone": "neutral",
            "summary": "",
            "visuals": ["none"],
            "narration_style": "natural"
        }
        
    # Chuẩn hóa về format SegmentAnalysis của mã nguồn máy tính
    final_obj = {
        "segment_id": req['segment_id'],
        "keywords": parsed_data.get("keywords", []),
        "scene": parsed_data.get("scene", "general"),
        "tone": parsed_data.get("tone", "neutral"),
        "summary": parsed_data.get("summary", ""),
        "suggested_visuals": parsed_data.get("visuals", []),
        "narration_style": parsed_data.get("narration_style", "natural")
    }
    
    results.append(final_obj)
    print(f"Kết quả: {final_obj['scene']} - {final_obj['tone']}")


In [ ]:
# 6. Xuất Dữ liệu tải về máy
output_file = "analysis.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã hoàn tất! Vui lòng tải file '{output_file}' về máy tính và đặt vào thư mục 'data/outputs/'.")